# EDA - Fault Detection Dataset
Checking the training data before we start building any models. Need to understand what we're working with.

## Import Libraries
Loading pandas for data manipulation, numpy for numerical operations, matplotlib & seaborn for visualizations, and scipy for statistical analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

## Loading the data
Reading TRAIN.csv and checking basic shape, memory usage.

In [ ]:
df = pd.read_csv('TRAIN.csv')

num_samples, num_features = df.shape
print(f'Shape: {num_samples} rows x {num_features} columns')
print(f'{num_features - 1} features (F01-F47), 1 target (Class)')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

## Missing values, duplicates, data types
Checking for null values, duplicate rows, and verifying all columns are numeric. Found 738 duplicates that need removal.

In [ ]:
print('Data types:')
print(df.dtypes.value_counts().to_string())

total_nulls = df.isnull().sum().sum()
print(f'\nMissing values: {total_nulls}')

duplicate_count = df.duplicated().sum()
print(f'Duplicates: {duplicate_count} ({duplicate_count/num_samples*100:.1f}%)')

if duplicate_count > 0:
    print(f'  -> will need to remove these before training')

## Basic stats
Computing mean, std, min, max, range, and IQR for all features to understand their distributions.

In [ ]:
stats = df.describe().T
stats['range'] = stats['max'] - stats['min']
stats['iqr'] = stats['75%'] - stats['25%']

stats[['mean', 'std', 'min', '50%', 'max', 'range', 'iqr']].round(4).head(15)

## Class distribution
Checking if the classes are balanced or not

In [ ]:
target_dist = df['Class'].value_counts().sort_index()
target_pct = df['Class'].value_counts(normalize=True).sort_index() * 100

imbalance_ratio = target_dist.max() / target_dist.min()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#3498db', '#e74c3c']
bars = ax1.bar(['Normal (0)', 'Fault (1)'], target_dist.values, color=colors, edgecolor='black')
for bar, count, pct in zip(bars, target_dist.values, target_pct.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 100,
             f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
ax1.set_ylabel('Count')
ax1.set_title('Class Distribution')

ax2.pie(target_dist.values, labels=['Normal', 'Fault'], autopct='%1.1f%%',
        colors=colors, startangle=90, explode=(0.03, 0.03))
ax2.set_title('Class Proportions')

plt.tight_layout()
plt.show()

print(f'Imbalance ratio: {imbalance_ratio:.2f}:1 - looks fairly balanced')

## Feature distributions
Plotting histograms for all 47 features to visualize their distributions and identify skewness patterns.

In [ ]:
feature_cols = [c for c in df.columns if c.startswith('F')]

n_cols = 6
n_rows = (len(feature_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    ax = axes[i]
    skew = df[feat].skew()
    ax.hist(df[feat], bins=40, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(df[feat].mean(), color='red', linestyle='--', linewidth=1.5)
    ax.set_title(f'{feat} (skew={skew:.2f})', fontsize=9)
    ax.grid(alpha=0.3)

for i in range(len(feature_cols), len(axes)):
    axes[i].axis('off')

plt.suptitle('All Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Skewness
Most features look really skewed, lets check how many. Calculating skewness values - 44 features have |skew| > 1, will need Yeo-Johnson transformation.

In [ ]:
skewness = df[feature_cols].skew().sort_values(ascending=False)

highly_skewed = skewness[skewness.abs() > 1]
print(f'Highly skewed (|skew| > 1): {len(highly_skewed)}/{len(feature_cols)}')
print(f'Moderate skew: {((skewness.abs() > 0.5) & (skewness.abs() <= 1)).sum()}')
print(f'Low skew: {(skewness.abs() <= 0.5).sum()}')

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if abs(x) > 1 else '#f39c12' if abs(x) > 0.5 else '#27ae60' for x in skewness]
ax.barh(range(len(skewness)), skewness.values, color=colors, alpha=0.7)
ax.set_yticks(range(len(skewness)))
ax.set_yticklabels(skewness.index, fontsize=8)
ax.axvline(0, color='black')
ax.axvline(-1, color='gray', linestyle='--', alpha=0.5)
ax.axvline(1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Skewness')
ax.set_title('Skewness per Feature')
plt.tight_layout()
plt.show()

# most features are heavily skewed -> will need Yeo-Johnson transform

## Correlation analysis
Looking for features that are basically copies of each other. Finding pairs with |r| > 0.9 to drop redundant features and reduce multicollinearity.

In [ ]:
corr_matrix = df[feature_cols].corr()

# find pairs with very high correlation
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], 
                                    corr_matrix.iloc[i, j]))

print(f'Found {len(high_corr_pairs)} pairs with |r| > 0.9:\n')
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
    print(f'  {f1} <-> {f2}: r = {r:.4f}')

# heatmap
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, square=True, linewidths=0.3, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

# looks like some features are mirror images of each other (F01-F29, F02-F22, etc)
# should drop one from each pair during preprocessing

## Outliers
Using IQR method to detect outliers in each feature. Will use Winsorization (1st-99th percentile capping) during preprocessing.

In [ ]:
outlier_info = []
for feat in feature_cols:
    Q1 = df[feat].quantile(0.25)
    Q3 = df[feat].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    n_outliers = ((df[feat] < lower) | (df[feat] > upper)).sum()
    if n_outliers > 0:
        outlier_info.append({'Feature': feat, 'Outliers': n_outliers, 
                            'Pct': round(n_outliers/num_samples*100, 2)})

outlier_df = pd.DataFrame(outlier_info).sort_values('Outliers', ascending=False)
print(f'Features with outliers: {len(outlier_df)}/{len(feature_cols)}')
print(f'Total outlier values: {outlier_df["Outliers"].sum():,}')
outlier_df.head(10)

## Summary
Final EDA summary with key findings and preprocessing steps needed: remove duplicates, cap outliers, transform skewed features, drop correlated features, and scale.

In [ ]:
print('EDA Summary')
print('-' * 40)
print(f'Samples: {num_samples:,}')
print(f'Features: {len(feature_cols)}')
print(f'Missing values: {total_nulls}')
print(f'Duplicates: {duplicate_count}')
print(f'Class 0: {target_dist[0]:,} ({target_pct[0]:.1f}%)')
print(f'Class 1: {target_dist[1]:,} ({target_pct[1]:.1f}%)')
print(f'Highly skewed: {len(highly_skewed)} features')
print(f'Correlated pairs (|r|>0.9): {len(high_corr_pairs)}')
print(f'Features with outliers: {len(outlier_df)}')
print()
print('Next steps:')
print('- Remove 738 duplicates')
print('- Cap outliers (Winsorization)')
print('- Yeo-Johnson transform for skewness')
print('- Drop redundant correlated features')
print('- StandardScaler')